## 연습문제 1-1

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise RuntimeError(".env파일에 OPEN_API_KEY가 없습니다.")
print("OPEN_API_KEY 설정 확인 완료")

OPEN_API_KEY 설정 확인 완료


In [2]:
from langchain_openai import ChatOpenAI
OPENAI_MODEL="gpt-4o-mini"
TEMPERATURE=0.7
def get_llm(model:str=OPENAI_MODEL,temperature:float=TEMPERATURE) -> ChatOpenAI:
    return ChatOpenAI(api_key=OPENAI_API_KEY, model=model,temperature=temperature)
    
    

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
prompt = PromptTemplate.from_template(
    """
당신은 초보자에게 쉽게 요리를 알려주는 AI 요리사입니다.

사용자가 가진 재료: {ingredients}

위 재료로 만들 수 있는 요리 한 가지와 간단한 레시피를 추천해주세요.
기름, 소금 등 기본 양념을 추가한다면 별도로 표시해주세요.
한국어로 다음 형식에 맞춰 답변해주세요.

추천 요리: 요리 이름
재료: 사용 재료와 대략적인 분량
추가 재료: 필요한 기본 양념 등
조리법:
1. 첫 번째 단계
2. 두 번째 단계
3. 세 번째 단계
필요하면 단계를 더 추가해주세요.
"""
)

In [4]:
parser=StrOutputParser()
model=get_llm()
chain=prompt|model|parser
ingredients=input("가지고 있는 재료를 입력하세요: ")
if ingredients:
    result=chain.invoke({"ingredients": ingredients})
    print("\n"+result)
    
else:
    print("한 가지 이상의 재료를 입력해주세요")


추천 요리: 김치볶음밥

재료: 
- 밥 1공기
- 계란 2개
- 김치 1컵 (잘게 썬 것)

추가 재료: 
- 식용유 2큰술
- 소금 약간
- 후추 약간
- 참기름 1작은술 (선택 사항)
- 대파 (선택 사항)

조리법:
1. 프라이팬에 식용유를 두르고 중불로 가열합니다.
2. 잘게 썬 김치를 넣고 2~3분간 볶아 김치의 맛을 살립니다.
3. 볶은 김치에 밥을 넣고 잘 섞어가며 3~5분간 볶습니다. 이때 소금과 후추로 간을 맞춥니다.
4. 다른 팬에 계란을 깨서 스크램블 형태로 익힙니다.
5. 볶음밥이 잘 익으면, 스크램블한 계란을 넣고 섞어줍니다.
6. 마지막으로 참기름을 뿌리고 잘 섞습니다. 대파를 넣고 싶다면 이 단계에서 추가하면 좋습니다.
7. 그릇에 담아 맛있게 즐기세요!


## 연습문제 1-2

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import SystemMessagePromptTemplate
from langchain_core.prompts import HumanMessagePromptTemplate

model=get_llm()
parser=StrOutputParser()


In [6]:
system_message1=SystemMessagePromptTemplate.from_template(
    "당신은 여행 전문가입니다. "
    "사용자가 입력한 도시나 국가의 대표 관광 명소를 한 가지 추천하세요. "
    "설명 없이 명소 이름만 한국어로 출력하세요."
)
user_message1=HumanMessagePromptTemplate.from_template(
    "여행지: {destination}"
)
prompt1=ChatPromptTemplate.from_messages([system_message1,user_message1])

In [8]:
system_message2=SystemMessagePromptTemplate.from_template(
    "당신은 관광 안내 가이드입니다. "
    "주어진 명소의 역사, 특징, 방문 팁을 한국어로 쉽게 설명하세요. "
    "확실하지 않은 사실은 추측하지 마세요. "
    "운영 시간과 요금 등 변경될 수 있는 정보는 공식 안내 확인을 권하세요."
)
user_message2=HumanMessagePromptTemplate.from_template(
        """
다음 명소에 대해 자세히 설명해주세요.

명소: {attraction}

다음 형식으로 작성해주세요.

명소: 명소 이름

역사:
1. ...
2. ...

특징:
1. ...
2. ...

방문 팁:
1. ...
2. ...
"""
)
prompt2=ChatPromptTemplate.from_messages([system_message2,user_message2])


In [9]:
chain1=prompt1|model|parser
chain2=prompt2|model|parser
full_chain=({"attraction":chain1}|RunnablePassthrough.assign(details=chain2))

In [10]:
destination=input("여행할 도시나 국가를 입력하세요: ").strip()
if destination:
    result=full_chain.invoke({"destination":destination})
    print("\n1단계 결과: ",result['attraction'])
    print("\n2단계 결과: ")
    print(result["details"])
    
else:
    print("여행지를 입력해주세요")


1단계 결과:  콜로세움

2단계 결과: 
명소: 콜로세움

역사:
1. 콜로세움은 로마 제국 시대인 70년에서 80년 사이에 건설되었습니다. 당시 황제 베스파시아누스의 명령으로 시작되어 그의 아들 티투스에 의해 완공되었습니다.
2. 이곳은 고대 로마의 전투와 검투사 경기를 위해 지어진 대형 원형 경기장으로, 약 5만 명에서 8만 명까지 수용할 수 있었습니다. 콜로세움은 로마의 상징으로 여겨지며, 유네스코 세계문화유산으로도 등록되어 있습니다.

특징:
1. 콜로세움은 지름 약 188미터, 높이 약 48미터의 거대한 구조물로, 고대 로마 건축의 뛰어난 기술력을 보여줍니다. 외관은 화산암과 콘크리트로 만들어졌으며, 80개의 입구를 통해 관객들이 쉽게 출입할 수 있도록 설계되었습니다.
2. 이곳에서는 검투사 경기뿐만 아니라 야생 동물 싸움, 해상 전투 등 다양한 오락이 펼쳐졌습니다. 또한, 콜로세움 내부에는 복잡한 지하 통로와 전시 공간이 있어, 이곳에서 여러 가지 사건들이 일어났던 역사적인 장소입니다.

방문 팁:
1. 콜로세움은 인기 있는 관광지이므로, 사전 예약을 통해 입장권을 구매하는 것이 좋습니다. 특히 성수기에는 긴 대기 줄이 형성될 수 있습니다.
2. 내부를 탐방할 때에는 가이드 투어를 이용하면 콜로세움의 역사와 재미있는 이야기를 더 잘 이해할 수 있습니다. 오디오 가이드도 제공되니 참고하세요.


## 연습문제 1-3

In [11]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
examples = [
    {
        "news": (
            "삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) "
            "가속기를 처음으로 출시할 예정이다. 이는 AI 반도체 시장에서 "
            "지배적인 위치를 차지하고 있는 엔비디아의 독점에 도전하고, "
            "세계 최고의 반도체 제조업체로서의 지위를 다시 확립하려는 "
            "삼성전자의 노력으로 해석된다."
        ),
        "keywords": "삼성전자, 인공지능, 엔비디아"
    },
    {
        "news": (
            "세계보건기구(WHO)는 최근 새로운 건강 위기에 대응하기 위해 "
            "국제 협력의 중요성을 강조했다. 전염병 대응 역량의 강화와 "
            "글로벌 보건 시스템의 개선이 필요하다고 발표했다."
        ),
        "keywords": "세계보건기구, 건강위기, 국제협력"
    },
    {
        "news": (
            "정부는 탄소 배출을 줄이기 위해 재생에너지 투자를 확대하고 "
            "태양광 발전 시설 설치를 지원하기로 했다."
        ),
        "keywords": "탄소배출, 재생에너지, 태양광"
    }
]
example_prompt=ChatPromptTemplate.from_messages([("human","{news}"),
                                                 ("ai","키워드: {keywords}")])


In [12]:
few_shot_prompt=FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

In [13]:
final_prompt=ChatPromptTemplate.from_messages([
    ("system",
     "당신은 뉴스 키워드 추출 전문가입니다. "
     "예시를 참고하여 뉴스에서 핵심 키워드를 정확히 3개 추출하세요. "
     "뉴스에 근거한 명사 또는 짧은 명사구를 사용하고, "
     "의미가 중복되는 키워드는 피하세요. "
     "반드시 '키워드: 키워드1, 키워드2, 키워드3' 형식으로 "
     "한 줄만 출력하세요. 추가 설명은 하지 마세요."), few_shot_prompt,
    ("human","{input}")
])

In [14]:
llm=get_llm(temperature=0)
chain=final_prompt|llm
test_news=(
    "제미나이 2.0 플래시는 현재 구글 AI 스튜디오(Google AI Studio) "
    "및 버텍스 AI(Vertex AI)에서 제미나이 API를 통해 개발자에게 "
    "실험 모델로 제공됩니다. 모든 개발자는 멀티모달 입력 및 "
    "텍스트 출력을 사용할 수 있으며, 텍스트 음성 변환"
    "(text-to-speech) 및 네이티브 이미지 생성은 일부 파트너들을 "
    "대상으로 제공됩니다. 내년 1월에는 더 많은 모델 사이즈와 "
    "함께 일반에 공개될 예정입니다."
)

In [15]:
result=chain.invoke({'input':test_news})
print("[기술 뉴스]")
print(result.content)

[기술 뉴스]
키워드: 제미나이, 구글 AI, 멀티모달
